In [1]:
import torch
torch.__version__

'2.5.1+cu124'

In [9]:
import numpy as np
import torch
import csv

torch.set_printoptions(edgeitems=2, precision=2, linewidth=75)

In [ ]:
# wine_path = "../data/p1ch4/tabular-wine/winequality-white.csv"
wine_path = "./data/p1ch4/tabular-wine/winequality-white.csv"
# 二维数组的类型(32 位浮点数)、用于分隔每行数据的分隔符以及不读取 第 1 行(因为它包含列名). 
wineq_numpy = np.loadtxt(wine_path, dtype=np.float32, delimiter=";", skiprows=1)
print(wineq_numpy.shape)
wineq_numpy

(4898, 12)


array([[ 7.  ,  0.27,  0.36, ...,  0.45,  8.8 ,  6.  ],
       [ 6.3 ,  0.3 ,  0.34, ...,  0.49,  9.5 ,  6.  ],
       [ 8.1 ,  0.28,  0.4 , ...,  0.44, 10.1 ,  6.  ],
       ...,
       [ 6.5 ,  0.24,  0.19, ...,  0.46,  9.4 ,  6.  ],
       [ 5.5 ,  0.29,  0.3 , ...,  0.38, 12.8 ,  7.  ],
       [ 6.  ,  0.21,  0.38, ...,  0.32, 11.8 ,  6.  ]], dtype=float32)

In [11]:
col_list = next(csv.reader(open(wine_path), delimiter=";"))

wineq_numpy.shape, col_list

((4898, 12),
 ['fixed acidity',
  'volatile acidity',
  'citric acid',
  'residual sugar',
  'chlorides',
  'free sulfur dioxide',
  'total sulfur dioxide',
  'density',
  'pH',
  'sulphates',
  'alcohol',
  'quality'])

In [12]:
wineq = torch.from_numpy(wineq_numpy)

wineq.shape, wineq.dtype

(torch.Size([4898, 12]), torch.float32)

In [14]:
data = wineq[:, :-1]  # 择所有行和除最后一列以外的所有列
data, data.shape

(tensor([[ 7.00,  0.27,  ...,  0.45,  8.80],
         [ 6.30,  0.30,  ...,  0.49,  9.50],
         ...,
         [ 5.50,  0.29,  ...,  0.38, 12.80],
         [ 6.00,  0.21,  ...,  0.32, 11.80]]),
 torch.Size([4898, 11]))

In [15]:
target = wineq[:, -1]  # 选择所有行和最后一列
target, target.shape

(tensor([6., 6.,  ..., 7., 6.]), torch.Size([4898]))

In [ ]:
# 如果我们想要将 target 张量转换为标签张量,
# 我们有 2 种方法, 这取决于我们使用分类数据的策略或目的. 一种是简单地将标签视为分数的整数向量
target = wineq[:, -1].long()
target

tensor([6, 6,  ..., 7, 6])

In [19]:
target_unsqueezed = target.unsqueeze(1)
target_unsqueezed

tensor([[6],
        [6],
        ...,
        [7],
        [6]])

In [ ]:
# 另一种方法是构建分数的一个独热编码(one-hot encoding),
# 即将 10 个分数分别编码到一个由 10 个元素组成的向量中, 除了其中一个元素设置为 1, 其他所有元素都设置为 0, 每个分数都有一个不同的索引.
target_onehot = torch.zeros(target.shape[0], 10)

# 使用 scatter_() 方法获得一个独热编码, 该方法将沿着参数提供的索引方向将源张量的值填充到输入张量中.
target_onehot.scatter_(1, target.unsqueeze(1), 1.0)

tensor([[0., 0.,  ..., 0., 0.],
        [0., 0.,  ..., 0., 0.],
        ...,
        [0., 0.,  ..., 0., 0.],
        [0., 0.,  ..., 0., 0.]])

In [ ]:
"""
张量 data, 它包含与化学特征分析相关的 11 个变量. 我们可以使用 PyTorch 张量 API 中的函数来处理张量表格数据.
让我们首先获得每列的平均值和标准差: 在本例中, dim=0 表示沿维度 0 执行规约.
此时, 我们可以通过减去平均值并除以标准差来对数据进行归一化, 这有助于学习过程.
"""
data_mean = torch.mean(data, dim=0)
data_mean

tensor([6.85e+00, 2.78e-01, 3.34e-01, 6.39e+00, 4.58e-02, 3.53e+01,
        1.38e+02, 9.94e-01, 3.19e+00, 4.90e-01, 1.05e+01])

In [24]:
data_var = torch.var(data, dim=0)
data_var

tensor([7.12e-01, 1.02e-02, 1.46e-02, 2.57e+01, 4.77e-04, 2.89e+02,
        1.81e+03, 8.95e-06, 2.28e-02, 1.30e-02, 1.51e+00])

In [25]:
data_normalized = (data - data_mean) / torch.sqrt(data_var)
data_normalized

tensor([[ 1.72e-01, -8.18e-02,  ..., -3.49e-01, -1.39e+00],
        [-6.57e-01,  2.16e-01,  ...,  1.34e-03, -8.24e-01],
        ...,
        [-1.61e+00,  1.17e-01,  ..., -9.63e-01,  1.86e+00],
        [-1.01e+00, -6.77e-01,  ..., -1.49e+00,  1.04e+00]])

In [ ]:
"""
让我们先分析数据, 看看是否有一种简单的方法可以快速分辨出好酒和劣质酒. \
首先, 我们要确定 target 中哪些行对应的分数小于或等于 3
注意, 只有 20 个 bad_indexes 记录项被设置为 True, 通过使用 PyTorch 中高级索引的功能, 我们可以使用数据类型 torch.bool 来索引张量 data.
这实际上是过滤张量 data, 使其仅包含索引张量中与 True 对应的项或行.
张量 bad_indexes 与张量 target 具有相同的形状, 其值为 True 或 False 取决于我们的阈值与原始张量 target 的比较结果.
注意, 新的张量 bad_data 有 20 行, 与张量 bad_indexes 中为 True 的行数相等, 它保留了所有列.
"""
bad_indexes = target <= 3  # PyTorch 还提供比较函数, 这里可以用 torch.le(target, 3), 但使用操作符似乎更标准些
bad_indexes.shape, bad_indexes.dtype, bad_indexes.sum()

(torch.Size([4898]), torch.bool, tensor(20))

In [29]:
bad_data = data[bad_indexes]
bad_data.shape

torch.Size([20, 11])

In [ ]:
# 我们可以开始把葡萄酒分为好酒、中等酒和劣质酒 3 类.
bad_data = data[target <= 3]
mid_data = data[(target > 3) & (target < 7)]  # 对于布尔型 NumPy 数组和 PyTorch 张量, & 操作符执行逻辑与操作
good_data = data[target >= 7]

bad_mean = torch.mean(bad_data, dim=0)
mid_mean = torch.mean(mid_data, dim=0)
good_mean = torch.mean(good_data, dim=0)

for i, args in enumerate(zip(col_list, bad_mean, mid_mean, good_mean)):
    print("{:2} {:20} {:6.2f} {:6.2f} {:6.2f}".format(i, *args))

 0 fixed acidity          7.60   6.89   6.73
 1 volatile acidity       0.33   0.28   0.27
 2 citric acid            0.34   0.34   0.33
 3 residual sugar         6.39   6.71   5.26
 4 chlorides              0.05   0.05   0.04
 5 free sulfur dioxide   53.33  35.42  34.55
 6 total sulfur dioxide 170.60 141.83 125.25
 7 density                0.99   0.99   0.99
 8 pH                     3.19   3.18   3.22
 9 sulphates              0.47   0.49   0.50
10 alcohol               10.34  10.26  11.42


In [31]:
total_sulfur_threshold = 141.83
total_sulfur_data = data[:, 6]
predicted_indexes = torch.lt(total_sulfur_data, total_sulfur_threshold)

predicted_indexes.shape, predicted_indexes.dtype, predicted_indexes.sum()

(torch.Size([4898]), torch.bool, tensor(2727))

In [32]:
actual_indexes = target > 5

actual_indexes.shape, actual_indexes.dtype, actual_indexes.sum()

(torch.Size([4898]), torch.bool, tensor(3258))

In [33]:
n_matches = torch.sum(actual_indexes & predicted_indexes).item()
n_predicted = torch.sum(predicted_indexes).item()
n_actual = torch.sum(actual_indexes).item()

n_matches, n_matches / n_predicted, n_matches / n_actual

(2018, 0.74000733406674, 0.6193984039287906)